## End-to-End Data Integrity Test Suite

#### **Objective**
To verify that data flows correctly from **Bronze** (Raw) &rarr; **Silver** (Clean) &rarr; **Gold** (Aggregated) without corruption or loss.

#### **Testing Strategy: "Business Key Hashing"**
Instead of hashing *every* column (which breaks due to metadata changes like `ingestion_timestamp`), we generate a **Composite Hash** using the immutable Business Key columns:
1.  **User ID** (Who)
2.  **Event Time** (When)
3.  **Product ID** (What)
4.  **Price** (Value)
5.  **Event Type** (Action)

If the **SHA-256 Hash** of these 5 columns matches across layers, we prove that the **Business Event** was preserved exactly as it occurred.


In [0]:
# =============================================================================
# INTEGRITY TEST SUITE: END-TO-END VERIFICATION
# Strategy: "Composite Business Key" Hashing
#
# We do NOT hash all columns (e.g., ingestion_timestamp, _rescued_data) 
# because metadata naturally changes between layers. 
# We hash the 5 immutable "Business Key" columns that define a unique event:
# (User + Time + Product + Price + Type)
# =============================================================================

from pyspark.sql.functions import col, sha2, concat_ws, sum as _sum

CATALOG = "ecommerce_analytics_dev"
BRONZE_TABLE = f"{CATALOG}.bronze_layer.events_raw"
SILVER_TABLE = f"{CATALOG}.silver_layer.events_silver"
GOLD_FACT    = f"{CATALOG}.gold_layer.gold_fact_sales"

print(f"Testing Integrity for: {CATALOG}")

# =============================================================================
# TEST 1: BRONZE vs. SILVER INTEGRITY
# Logic: Verify valid Bronze rows exist in Silver.
# =============================================================================

# 1. Load Bronze (Apply same validity filter as the Silver Pipeline)
# We cast types here to match Silver's schema enforcement
df_bronze = spark.read.table(BRONZE_TABLE) \
    .filter("user_id IS NOT NULL AND event_time IS NOT NULL AND price >= 0") \
    .withColumn("integrity_hash", 
        sha2(concat_ws("||", 
            col("user_id").cast("integer"), 
            col("event_time").cast("timestamp"), 
            col("product_id").cast("integer"), 
            col("price").cast("double"),
            col("event_type") 
        ), 256)
    ).select("integrity_hash")

# 2. Load Silver
df_silver = spark.read.table(SILVER_TABLE) \
    .withColumn("integrity_hash", 
        sha2(concat_ws("||", 
            col("user_id"), 
            col("event_time"), 
            col("product_id"), 
            col("price"),
            col("event_type")
        ), 256)
    ).select("integrity_hash")

# 3. Compare Counts
matched_count = df_bronze.join(df_silver, on="integrity_hash", how="inner").count()
bronze_count = df_bronze.count()
silver_count = df_silver.count()

print(f"Bronze Rows (Valid): {bronze_count}")
print(f"Silver Rows:         {silver_count}")
print(f"Matched Integrity:   {matched_count}")

# Silver performs deduplication, so Silver count <= Bronze Count is expected.
# We pass if we have significant overlap.
if matched_count > 0 and matched_count >= silver_count:
    print(" TEST PASSED: Bronze -> Silver Data Integrity Verified.")
else:
    print(f" TEST FAILED: Integrity issues found.")

In [0]:
# =============================================================================
# TEST 2: SILVER vs. GOLD FACT INTEGRITY
# Logic: Verify Silver transactions appear in Gold Fact.
# Note: Gold maps 'transaction_amount' back to 'price'.
# =============================================================================

# 1. Load Silver Hash
df_silver_chk = spark.read.table(SILVER_TABLE) \
    .withColumn("integrity_hash", 
        sha2(concat_ws("||", 
            col("user_id"), 
            col("event_time"), 
            col("product_id"), 
            col("price"),
            col("event_type") 
        ), 256)
    ).select("integrity_hash")

# 2. Load Gold Hash 
# CRITICAL: We use 'transaction_amount' because 'price' was renamed in Gold.
df_gold_chk = spark.read.table(GOLD_FACT) \
    .withColumn("integrity_hash", 
        sha2(concat_ws("||", 
            col("user_id"), 
            col("event_time"), 
            col("product_id"), 
            col("transaction_amount"), 
            col("event_type")
        ), 256)
    ).select("integrity_hash")

# 3. Compare
gold_matches = df_silver_chk.join(df_gold_chk, on="integrity_hash", how="inner").count()
silver_count = df_silver_chk.count()

print(f"Silver Rows:     {silver_count}")
print(f"Gold Matches:    {gold_matches}")

if gold_matches >= silver_count: 
    print(" TEST PASSED: Silver -> Gold Data Integrity Verified.")
else:
    print(f" Note: Gold might be missing {silver_count - gold_matches} rows (Expected if Gold logic aggregates or filters).")

In [0]:

# =============================================================================
# TEST 3: FINANCIAL REVENUE CHECK (The "CFO" Test)
# =============================================================================
silver_rev = spark.read.table(SILVER_TABLE).agg(_sum("price")).collect()[0][0]
gold_rev   = spark.read.table(GOLD_FACT).agg(_sum("transaction_amount")).collect()[0][0]

# Handle potential None/Null values for safety
silver_rev = silver_rev if silver_rev else 0.0
gold_rev = gold_rev if gold_rev else 0.0

diff = abs(silver_rev - gold_rev)

print(f"Silver Revenue: ${silver_rev:,.2f}")
print(f"Gold Revenue:   ${gold_rev:,.2f}")

if diff < 1.0:
    print("TEST PASSED: 100% Financial Integrity.")
else:
    print(f" TEST FAILED: Revenue Mismatch by ${diff:,.2f}")